In [ ]:
import json
from typing import Dict
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import ast
from constants_and_path_utils import RADAGENT_RESULTS_DIR

VALID_TOOLS = [
        "report_generation_tool",
        "disease_classifier_tool",
        "ct_vqa_tool",
        "extract_slices_from_ct",
        "slice_vqa_tool",
        "anatomy_segmentation_tool",
        "effusion_segmentation_tool",
        "biggest_slice_selection_tool",
        "best_slice_selection_tool",
        "get_several_slices_from_segmentation",
        "get_several_slices_tool",
        "three_equidistant_slice_selection_tool",
        "windowing_tool",
    ]

def get_tool_calls(data):
    tool_called = []
    success_tool_called = []
    fail_tool_called = []
    last_tool_called = None
    reward = 0
    coherence = 0
    diversity = 0
    checklist_adherence = 0
    for message in data:
        if isinstance(message, Dict):
            try:
                if message["role"] == "assistant":
                    response = json.loads(message["content"])
                    if response.get("action", None) == "call_tool":
                        last_tool_called = response["tool_name"]
                        tool_called.append(last_tool_called)
                if message["role"] == "tool":
                    if (
                        message["content"] == "Tool call failed"
                        or ("ERROR" in message["content"].upper())
                        or last_tool_called is None
                    ):
                        fail_tool_called.append(last_tool_called)
                    else:
                        success_tool_called.append(last_tool_called)
                        last_tool_called = None
            except (json.JSONDecodeError, KeyError):
                if message.get("reward", None) is not None:
                    reward = message["reward"]
                if message.get("tool sequence coherence", None) is not None:
                    coherence = message["tool sequence coherence"]["score"]
                if message.get("tool diversity", None) is not None:
                    diversity = message["tool diversity"]["score"]
                if message.get("checklist adherence", None) is not None:
                    checklist_adherence = message["checklist adherence"]["score"]
                else:
                    continue
    return (
        tool_called,
        fail_tool_called,
        success_tool_called,
        reward,
        coherence,
        diversity,
        checklist_adherence,
    )


def get_tool_counts_df(df, column_name):
    """
    Parses a column of stringified lists and returns a DataFrame
    of counts for each tool found in that column.
    """

    def parse_list_safe(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []
        return x if isinstance(x, list) else []

    # Parse the strings into lists
    parsed_series = df[column_name].apply(parse_list_safe)

    # Create counts dataframe
    counts_df = (
        pd.DataFrame(parsed_series.apply(Counter).tolist()).fillna(0).astype(int)
    )

    # attach image_id for tracking (useful if we need to merge later, though index is usually sufficient)
    counts_df["image_id"] = df["image_id"]
    return counts_df


def plot_metric(data_df, metric_name, title, y_label, y_limit=None, savepath=None):
    """
    Generic function to plot bar charts with error bars using Seaborn.
    """
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
    plt.rcParams["font.size"] = 10
    plt.rcParams["axes.labelsize"] = 12
    plt.rcParams["axes.titlesize"] = 12
    plt.rcParams["xtick.labelsize"] = 10
    plt.rcParams["ytick.labelsize"] = 10
    plt.rcParams["legend.fontsize"] = 9
    plt.rcParams["axes.linewidth"] = 0.8

    # Use white background, no grid
    sns.set_style("white")
    
    # Melt for Seaborn
    df_melted = data_df.melt(
        id_vars=["image_id"], var_name="Tool", value_name=metric_name
    )

    # Drop NaNs (important for Success Rate where 0/0 is NaN)
    df_melted = df_melted.dropna(subset=[metric_name])

    if df_melted.empty:
        print(f"No data available for {title}")
        return

    # Determine sort order
    order = (
        df_melted.groupby("Tool")[metric_name].mean().sort_values(ascending=False).index
    )

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(
        data=df_melted,
        x="Tool",
        y=metric_name,
        #errorbar=("sd", 1),
        order=order,
        palette="Blues_r",
        edgecolor="black",
        capsize=0.1,
    )
    # Remove top and right spines (Nature style)
    sns.despine(ax=ax, top=True, right=True)

    # Add horizontal grid lines for readability
    ax.yaxis.grid(True, linestyle="-", linewidth=0.5, color="lightgray", alpha=0.7)
    ax.set_axisbelow(True)  # Grid behind bars
    
    plt.title(title, fontweight="bold")
    plt.ylabel(y_label)
    plt.xlabel("Tool Name")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    if y_limit:
        plt.ylim(y_limit)
    if savepath is not None:
        plt.savefig(savepath, format="pdf", bbox_inches="tight", dpi=300)
        print(f"Figure saved to {savepath}")
    plt.show()



In [ ]:
# list all files in directory
directory = RADAGENT_RESULTS_DIR / "inference_directory_with_trajectory_jsons"
# directory can point to any saved inference folder that contains trajectory JSON files
pathlist = list(Path(directory).glob("**/*.json"))

valid_ids = None
image_id = []
all_tool_calls = []
all_failed_tool_calls = []
all_success_tool_calls = []
rewards = []
all_coherences = []
all_diversities = []
all_checklist_adherences = []
for path in pathlist:
    c_image_id = path.stem.split("_trajectory")[0]

    if (valid_ids is None) or (c_image_id in valid_ids):
        with open(path, "r") as f:
            data = json.load(f)
        # try:
        (
            tool_called,
            fail_tool_called,
            success_tool_called,
            reward,
            coherence,
            diversity,
            checklist_adherence,
        ) = get_tool_calls(data)
        # except Exception as e:
        #     continue
        all_tool_calls.append(tool_called)
        all_failed_tool_calls.append(fail_tool_called)
        all_success_tool_calls.append(success_tool_called)
        all_coherences.append(coherence)
        all_diversities.append(diversity)
        all_checklist_adherences.append(checklist_adherence)
        image_id.append(c_image_id)
        rewards.append(reward)

df = pd.DataFrame(
    {
        "image_id": image_id,
        "tool_calls": all_tool_calls,
        "successful_tool_calls": all_success_tool_calls,
        "failed_tool_calls": all_failed_tool_calls,
        "reward": [r + 0.3 for r in rewards],
        "coherence": all_coherences,
        "diversity": all_diversities,
        "checklist_adherence": all_checklist_adherences,
        "traj_path": pathlist,
    }
)
print(len(df))

In [ ]:
# --- 1. Calculate Counts for all categories ---
def plot_tool_use_analysis(df):
    df_total_counts = get_tool_counts_df(df, "tool_calls")
    df_success_counts = get_tool_counts_df(df, "successful_tool_calls")
    df_failed_counts = get_tool_counts_df(df, "failed_tool_calls")

    # --- 2. Align Columns (Crucial Step) ---
    # Ensure Success and Failed DFs have all columns present in Total DFs
    # (e.g., if a tool never failed, the failed DF needs a column of 0s for it)
    all_tools = [c for c in df_total_counts.columns if c != "image_id"]

    
    df_success_aligned = df_success_counts.reindex(columns=all_tools, fill_value=0)
    df_success_aligned["image_id"] = df["image_id"]

    df_failed_aligned = df_failed_counts.reindex(columns=all_tools, fill_value=0)
    df_failed_aligned["image_id"] = df["image_id"]

    # --- 3. Calculate Success Rate ---
    # Rate = Success / Total
    # We drop image_id temporarily for division, then add it back
    # NOTE: 0/0 will result in NaN. This is desired. If a tool wasn't used in an image,
    # it shouldn't count towards the average rate as 0 or 1.
    df_rates = df_success_aligned[all_tools] / (
        df_total_counts[all_tools]
    )  # small epsilon to avoid division by zero
    df_rates = df_rates[[c for c in df_rates.columns if c in VALID_TOOLS]]
    df_rates["image_id"] = df["image_id"]
    # --- 4. Plotting ---

    # Plot A: All Calls
    plot_metric(
        df_total_counts,
        metric_name="Count",
        title="Average Tool Calls per Image",
        y_label="Avg Count",
        y_limit=(0, 12),
    )

    # Plot C: Success Rate
    plot_metric(
        df_rates,
        metric_name="Rate",
        title="Trained CTRadAgent\nAverage Success Rate per Tool",
        y_label="Success Rate (0-1)",
        y_limit=(0, 1.05),  # Fix y-axis to standard probability range
        savepath="tool_success_rates.pdf",
    )


plot_tool_use_analysis(df)

In [ ]:
import numpy as np
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D


def plot_workflow_sequences(
    df,
    top_n=15,
    min_count=5,
    title="Most Common Tool-Calling Workflows",
    savepath=None,
):
    """
    Visualise the top-N most frequent tool-calling workflows as a
    horizontal step grid.  Each row is a unique workflow, each column
    is a step index, and cells are colour-coded by tool.  The count /
    percentage of trajectories following each workflow is shown on the
    right-hand side.

    Non-interactive, pure matplotlib — suitable for PDF export.
    """

    # ── Abbreviations & colour map ──────────────────────────────────
    abbreviated_tool = {
        "slice_vqa_tool": "SliceVQA",
        "anatomy_segmentation_tool": "AnatomySeg",
        "biggest_slice_selection_tool": "BiggestSlice",
        "best_slice_selection_tool": "BestSlice",
        "disease_classifier_tool": "DiseaseClass",
        "report_generation_tool": "ReportGen",
        "ct_vqa_tool": "CT-VQA",
        "windowing_tool": "Windowing",
        "effusion_segmentation_tool": "EffusionSeg",
        "three_equidistant_slice_selection_tool": "3EquiSlice",
        "get_several_slices_tool": "MultiSliceSeg",
        "get_several_slices_from_segmentation": "MultiSliceSeg",
        "extract_slices_from_ct": "ExtractSlice",
    }

    tool_color_map = {
        "ReportGen":    "#e74c3c",
        "DiseaseClass": "#f39c12",
        "CT-VQA":       "#3498db",
        "ExtractSlice": "#2ecc71",
        "SliceVQA":     "#9b59b6",
        "AnatomySeg":   "#1abc9c",
        "EffusionSeg":  "#16a085",
        "BiggestSlice": "#2980b9",
        "BestSlice":    "#2980b9",
        "3EquiSlice":   "#8e44ad",
        "MultiSliceSeg":"#27ae60",
        "Windowing":    "#d35400",
    }

    # ── Filter to non-empty workflows & count ────────────────────────
    s = df["tool_calls"].apply(
        lambda x: tuple(x) if isinstance(x, list) else ()
    )
    s = s[s.map(len) > 0]
    counts = s.value_counts()
    counts = counts[counts >= min_count].head(top_n)

    if counts.empty:
        print("No workflows above the minimum count threshold.")
        return

    total = len(s)
    workflows = counts.index.tolist()        # list of tuples
    workflow_counts = counts.values            # array of ints
    n_workflows = len(workflows)
    max_steps = max(len(w) for w in workflows)

    # ── Abbreviate tool names in each workflow ────────────────────────
    workflows_abbr = [
        [abbreviated_tool.get(t, t) for t in wf] for wf in workflows
    ]

    # ── Figure setup ─────────────────────────────────────────────────
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
    sns.set_style("white")

    cell_w = 1.0
    right_margin = 3.0     # space for count annotation
    row_gap = 0.08         # vertical gap between rows

    # ── Compute proportional row heights ──────────────────────────
    # Scale row heights so the total figure height is reasonable.
    # Each row height is proportional to its count.
    total_canvas_height = 10.0        # total usable vertical space
    min_row_h = 0.35                  # minimum so tiny rows stay readable
    total_gap = row_gap * (n_workflows - 1)
    usable_height = total_canvas_height - total_gap

    # Raw proportional heights
    raw_heights = np.array([float(c) for c in workflow_counts])
    raw_heights = raw_heights / raw_heights.sum() * usable_height
    # Enforce minimum, then re-normalise the remainder
    clamped = np.maximum(raw_heights, min_row_h)
    excess = clamped.sum() - usable_height
    if excess > 0:
        # Shrink only the rows that are above minimum proportionally
        above_min = clamped - min_row_h
        above_total = above_min.sum()
        if above_total > 0:
            clamped = clamped - above_min / above_total * excess
            clamped = np.maximum(clamped, min_row_h)
    row_heights = clamped  # array of per-row heights (top-to-bottom order)

    fig_w = max_steps * cell_w + right_margin + 1.5
    fig_h = total_canvas_height + 2.5
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # ── Pre-compute y-positions (bottom of each row, top-to-bottom) ──
    # Row 0 is the most common → drawn at the top
    row_y_bottom = np.zeros(n_workflows)
    y_cursor = total_canvas_height  # start from top
    for row_idx in range(n_workflows):
        y_cursor -= row_heights[row_idx]
        row_y_bottom[row_idx] = y_cursor
        y_cursor -= row_gap

    # ── Draw cells ────────────────────────────────────────────────────
    for row_idx, (wf, cnt) in enumerate(
        zip(workflows_abbr, workflow_counts)
    ):
        y_bot = row_y_bottom[row_idx]
        rh = row_heights[row_idx]
        for step_idx, tool in enumerate(wf):
            color = tool_color_map.get(tool, "#bdc3c7")
            pad = 0.04
            rect = FancyBboxPatch(
                (step_idx * cell_w + pad, y_bot + pad),
                cell_w - 2 * pad,
                rh - 2 * pad,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                facecolor=color,
                edgecolor="white",
                linewidth=0.8,
            )
            ax.add_patch(rect)

            # Draw arrows between consecutive steps
            if step_idx < len(wf) - 1:
                ax.annotate(
                    "",
                    xy=((step_idx + 1) * cell_w + pad, y_bot + rh / 2),
                    xytext=(step_idx * cell_w + cell_w - pad, y_bot + rh / 2),
                    arrowprops=dict(
                        arrowstyle="->",
                        color="#7f8c8d",
                        lw=0.8,
                        shrinkA=0,
                        shrinkB=0,
                    ),
                )

        # Count / percentage label on the right
        pct = cnt / total * 100
        ax.text(
            max_steps * cell_w + 0.3,
            y_bot + rh / 2,
            f"n={cnt}  ({pct:.1f}%)",
            va="center",
            ha="left",
            fontsize=8,
            fontweight="medium",
            color="#2c3e50",
        )

    # ── Step labels along top ─────────────────────────────────────────
    for s_idx in range(max_steps):
        ax.text(
            s_idx * cell_w + cell_w / 2,
            total_canvas_height + 0.15,
            f"Step {s_idx + 1}",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#7f8c8d",
        )

    # ── Legend ─────────────────────────────────────────────────────────
    # Collect tools that actually appear
    used_tools = sorted(
        {t for wf in workflows_abbr for t in wf},
        key=lambda t: list(tool_color_map.keys()).index(t)
        if t in tool_color_map else 999,
    )
    legend_handles = [
        Line2D(
            [0], [0],
            marker="s",
            color="w",
            markerfacecolor=tool_color_map.get(t, "#bdc3c7"),
            markersize=8,
            markeredgecolor="white",
            label=t,
        )
        for t in used_tools
    ]
    ax.legend(
        handles=legend_handles,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=False,
        fontsize=8,
        title="Tools",
        title_fontsize=9,
        handletextpad=0.4,
        labelspacing=0.3,
    )

    # ── Axes cosmetics ─────────────────────────────────────────────────
    ax.set_xlim(-0.1, max_steps * cell_w + right_margin)
    ax.set_ylim(-0.2, total_canvas_height + 0.6)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=16)

    plt.tight_layout()
    if savepath is not None:
        plt.savefig(savepath, format="pdf", bbox_inches="tight", dpi=300)
        print(f"Figure saved to {savepath}")
    plt.show()


plot_workflow_sequences(
    df,
    top_n=13,
    min_count=5,
    title="Most Common Tool-Calling Workflows",
    savepath="tool_workflows.pdf",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.path as mpath
from matplotlib.lines import Line2D
from collections import defaultdict, Counter


def plot_sankey_matplotlib(
    df,
    min_flow_pct=0.01,
    title="Tool Transition Flows Across Agent Steps",
    savepath=None,
):
    """
    Publication-quality Sankey diagram rendered entirely in matplotlib.

    Only sequences (complete workflows) that appear in at least
    ``min_flow_pct`` of all trajectories are included.  This keeps the
    number of step-columns consistent with the workflow grid plot and
    removes noise from rare long outlier trajectories.

    Parameters
    ----------
    df : DataFrame
        Must contain a ``tool_calls`` column of lists.
    min_flow_pct : float
        Minimum fraction of total sequences a *workflow* must represent
        to be included (e.g. 0.01 = 1%).
    title : str
        Figure title.
    savepath : str | None
        If given, save to this path as PDF.
    """

    # ── Abbreviations & colours ───────────────────────────────────────
    ABBR = {
        "slice_vqa_tool": "SliceVQA",
        "anatomy_segmentation_tool": "AnatomySeg",
        "biggest_slice_selection_tool": "BiggestSlice",
        "best_slice_selection_tool": "BestSlice",
        "disease_classifier_tool": "DiseaseClassifier",
        "report_generation_tool": "ReportGenerator",
        "ct_vqa_tool": "CT-VQA",
        "windowing_tool": "Windowing",
        "effusion_segmentation_tool": "EffusionSeg",
        "three_equidistant_slice_selection_tool": "3EquiSlice",
        "get_several_slices_tool": "MultiSliceSeg",
        "get_several_slices_from_segmentation": "MultiSliceSeg",
        "extract_slices_from_ct": "ExtractSlice",
    }

    COLORS = {
        "ReportGenerator":     "#084081",
        "DiseaseClassifier":  "#b1efa3",
        "CT-VQA":        "#2b8cbe",
        "ExtractSlice":  "#4ed383",
        "SliceVQA":      "#435647",
        "AnatomySeg":    "#a8ddb5",
        "EffusionSeg":   "#98a58f",
        "BiggestSlice":  "#3690c0",
        #"BestSlice":     "#43a2ca",
        #"3EquiSlice":    "#ccebc5",
        "MultiSliceSeg": "#d0d1e6",
        "Windowing":     "#f0f9e8",
    }

    # Canonical ordering so every column stacks tools in the same order
    TOOL_ORDER = [
        "DiseaseClassifier", "CT-VQA", "ExtractSlice", "SliceVQA",
        "AnatomySeg", "EffusionSeg", "BiggestSlice", "MultiSliceSeg", "Windowing", "ReportGenerator",
    ]

    # ── Pre-filter sequences to >= min_flow_pct ──────────────────────
    all_seqs = [
        tuple(ABBR.get(t, t) for t in row)
        for row in df["tool_calls"]
        if isinstance(row, list) and len(row) > 0
    ]
    n_total_seqs = len(all_seqs)
    seq_counts = Counter(all_seqs)
    min_count = max(1, int(n_total_seqs * min_flow_pct))
    seqs = [list(s) for s in all_seqs if seq_counts[s] >= min_count]

    if not seqs:
        print("No workflows above the minimum frequency threshold.")
        return

    n_kept = len(seqs)
    n_unique = len({tuple(s) for s in seqs})
    max_steps = max(len(s) for s in seqs)
    print(f"Sankey: kept {n_kept}/{n_total_seqs} trajectories ",
          f"({n_unique} unique workflows, max {max_steps} steps)")

    # ── Build per-step node sizes & transition counts ─────────────────
    node_counts = defaultdict(int)
    edge_counts = defaultdict(int)

    for seq in seqs:
        for i, tool in enumerate(seq):
            node_counts[(i, tool)] += 1
            if i < len(seq) - 1:
                edge_counts[(i, tool, seq[i + 1])] += 1

    # ── Geometry constants ────────────────────────────────────────────
    col_spacing = 1.8
    node_width = 0.22
    node_pad = 0.06
    total_height = 10.0

    # ── Compute node positions ────────────────────────────────────────
    node_pos = {}

    # Count how many sequences are active at each step
    step_totals = {}
    for step in range(max_steps):
        tools_here = [t for t in TOOL_ORDER if (step, t) in node_counts]
        extras = sorted(
            t for s, t in node_counts if s == step and t not in TOOL_ORDER
        )
        tools_here += extras
        step_totals[step] = sum(node_counts[(step, t)] for t in tools_here)

    max_total = max(step_totals.values()) if step_totals else 1

    for step in range(max_steps):
        tools_here = [t for t in TOOL_ORDER if (step, t) in node_counts]
        extras = sorted(
            t for s, t in node_counts if s == step and t not in TOOL_ORDER
        )
        tools_here += extras

        total_count = step_totals[step]
        if total_count == 0:
            continue

        # Scale column height proportionally to how many sequences are
        # still active at this step vs. the busiest step (usually step 0).
        col_height = total_height * (total_count / max_total)
        usable = col_height - node_pad * max(len(tools_here) - 1, 0)
        # Centre the (shorter) column vertically within the canvas
        y_offset = (total_height - col_height) / 2.0
        y_cursor = y_offset + col_height  # start from top of this column
        x_left = step * col_spacing

        for t in tools_here:
            h = (node_counts[(step, t)] / total_count) * usable
            y_cursor -= h
            node_pos[(step, t)] = (x_left, y_cursor, h)
            y_cursor -= node_pad

    # ── Helper: Bezier band ───────────────────────────────────────────
    def _bezier_band(ax, x0, y0_bot, y0_top, x1, y1_bot, y1_top,
                     color, alpha=0.35):
        xm = (x0 + x1) / 2.0
        verts = [
            (x0, y0_bot),
            (xm, y0_bot), (xm, y1_bot), (x1, y1_bot),
            (x1, y1_top),
            (xm, y1_top), (xm, y0_top), (x0, y0_top),
            (x0, y0_bot),
        ]
        codes = [
            mpath.Path.MOVETO,
            mpath.Path.CURVE4, mpath.Path.CURVE4, mpath.Path.CURVE4,
            mpath.Path.LINETO,
            mpath.Path.CURVE4, mpath.Path.CURVE4, mpath.Path.CURVE4,
            mpath.Path.CLOSEPOLY,
        ]
        patch = mpatches.PathPatch(
            mpath.Path(verts, codes),
            facecolor=color, edgecolor="none", alpha=alpha, lw=0,
        )
        ax.add_patch(patch)

    # ── Edge slot cursors (stack bands within each node) ──────────────
    out_cursor = {}
    in_cursor = {}
    for key, (xl, yb, h) in node_pos.items():
        out_cursor[key] = yb
        in_cursor[key] = yb

    sorted_edges = sorted(edge_counts.items(), key=lambda kv: -kv[1])

    # ── Figure ────────────────────────────────────────────────────────
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]

    fig_w = max_steps * col_spacing + 2.5
    fig_h = total_height * 0.65
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # ── Draw edges ────────────────────────────────────────────────────
    for (step, src, dst), count in sorted_edges:
        src_key = (step, src)
        dst_key = (step + 1, dst)
        if src_key not in node_pos or dst_key not in node_pos:
            continue

        src_xl, src_yb, src_h = node_pos[src_key]
        dst_xl, dst_yb, dst_h = node_pos[dst_key]

        band_h_src = (count / node_counts[src_key]) * src_h
        band_h_dst = (count / node_counts[dst_key]) * dst_h

        y0_bot = out_cursor[src_key]
        y0_top = y0_bot + band_h_src
        out_cursor[src_key] = y0_top

        y1_bot = in_cursor[dst_key]
        y1_top = y1_bot + band_h_dst
        in_cursor[dst_key] = y1_top

        _bezier_band(
            ax,
            src_xl + node_width, y0_bot, y0_top,
            dst_xl, y1_bot, y1_top,
            color=COLORS.get(src, "#bdc3c7"), alpha=0.38,
        )

    # ── Draw nodes ────────────────────────────────────────────────────
    for (step, tool), (xl, yb, h) in node_pos.items():
        color = COLORS.get(tool, "#bdc3c7")
        rect = mpatches.FancyBboxPatch(
            (xl, yb), node_width, h,
            boxstyle="round,pad=0.01,rounding_size=0.03",
            facecolor=color, edgecolor="white", linewidth=0.6,
        )
        ax.add_patch(rect)
        if h > 0.25:
            ax.text(
                xl + node_width + 0.06, yb + h / 2,
                tool, va="center", ha="left",
                fontsize=12, fontweight="medium", color="#2c3e50",
            )

    # ── Step column headers ───────────────────────────────────────────
    for step in range(max_steps):
        ax.text(
            step * col_spacing + node_width / 2,
            total_height + 0.35,
            f"Step {step + 1}",
            ha="center", va="bottom",
            fontsize=12, fontweight="bold", color="#34495e",
        )

    # ── Legend ─────────────────────────────────────────────────────────
    used = sorted(
        {tool for (_, tool) in node_pos},
        key=lambda t: TOOL_ORDER.index(t) if t in TOOL_ORDER else 999,
    )
    handles = [
        Line2D(
            [0], [0], marker="s", color="w",
            markerfacecolor=COLORS.get(t, "#bdc3c7"),
            markersize=7, markeredgecolor="white", label=t,
        )
        for t in used
    ]
    ax.legend(
        handles=handles, loc="upper center",
        bbox_to_anchor=(0.5, -0.02), frameon=False,
        fontsize=18, title=None,
        ncol=len(used),
        handletextpad=0.3, labelspacing=0.25,
        columnspacing=1.0,
    )

    # ── Axes cosmetics ────────────────────────────────────────────────
    ax.set_xlim(-0.3, max_steps * col_spacing + 0.3)
    ax.set_ylim(-0.3, total_height + 0.8)
    ax.axis("off")
    ax.set_title(title, fontsize=20, fontweight="bold", pad=14)

    plt.tight_layout()
    if savepath is not None:
        fig.savefig(savepath, format="pdf", bbox_inches="tight", dpi=300)
        print(f"Figure saved to {savepath}")
    plt.show()


plot_sankey_matplotlib(
    df,
    min_flow_pct=0.01,
    title="Tool Transition Flows Across Agent Steps",
    savepath="tool_sankey.pdf",
)

In [ ]:
def print_most_common_tool_sequence_subset(df):
    """Find the most common tool sequence and print the subset of df that matches it."""
    # Convert tool_calls lists to tuples for counting
    sequences = df["tool_calls"].apply(
        lambda x: tuple(x) if isinstance(x, list) else ()
    )
    most_common_seq = sequences.value_counts().idxmax()
    print(f"Most common tool sequence ({sequences.value_counts().iloc[0]} occurrences):")
    print(f"  {list(most_common_seq)}\n")

    mask = sequences == most_common_seq
    subset = df[mask]
    print(f"Subset of df with the most common tool sequence ({len(subset)} rows):")
    display(subset)
    return subset

most_common_subset = print_most_common_tool_sequence_subset(df)

In [ ]:
# Randomly sample 2 trajectories from the most common subset and print their JSON
import random

sampled = most_common_subset.sample(n=2, random_state=random.randint(0, 10000))

for idx, row in sampled.iterrows():
    traj_path = row["traj_path"]
    print(f"\n{'='*80}")
    print(f"Image ID: {row['image_id']}")
    print(f"Trajectory path: {traj_path}")
    print(f"{'='*80}\n")
    with open(traj_path, "r") as f:
        traj_data = json.load(f)
    assistant_messages = [m for m in traj_data if isinstance(m, dict) and m.get("role") == "assistant"]
    filtered = []
    for m in assistant_messages:
        try:
            content = json.loads(m["content"])
            entry = {}
            if "tool_name" in content:
                entry["tool_name"] = content["tool_name"]
            if "arguments" in content:
                entry["arguments"] = content["arguments"]
            if entry:
                filtered.append(entry)
        except (json.JSONDecodeError, KeyError):
            continue
    print(json.dumps(filtered, indent=2))

In [ ]:
pd.read_csv(RADAGENT_RESULTS_DIR / 'detailed_results.csv')

In [ ]:
# Filter df to only show rows that use slice_vqa_tool
slice_vqa_df = df[df["tool_calls"].apply(lambda x: "slice_vqa_tool" in x if isinstance(x, list) else False)]
print(f"Rows using slice_vqa_tool: {len(slice_vqa_df)} / {len(df)}")
display(slice_vqa_df)
most_common_subset = print_most_common_tool_sequence_subset(slice_vqa_df)

In [ ]:

# Randomly sample 2 trajectories from the most common subset and print their JSON
import random

sampled = most_common_subset.sample(n=2, random_state=random.randint(0, 10000))

for idx, row in sampled.iterrows():
    traj_path = row["traj_path"]
    print(f"\n{'='*80}")
    print(f"Image ID: {row['image_id']}")
    print(f"Trajectory path: {traj_path}")
    print(f"{'='*80}\n")
    with open(traj_path, "r") as f:
        traj_data = json.load(f)
    assistant_messages = [m for m in traj_data if isinstance(m, dict) and m.get("role") == "assistant"]
    filtered = []
    for m in assistant_messages:
        try:
            content = json.loads(m["content"])
            entry = {}
            if "tool_name" in content:
                entry["tool_name"] = content["tool_name"]
            if "arguments" in content:
                entry["arguments"] = content["arguments"]
            if entry:
                filtered.append(entry)
        except (json.JSONDecodeError, KeyError):
            continue
    print(json.dumps(filtered, indent=2))

In [ ]:
traj_data

In [ ]:
longer_vqa_df = df[df["tool_calls"].apply(lambda x: len(x) > 12 if isinstance(x, list) else False)]
print(f"Rows with more than 12 tool calls: {len(longer_vqa_df)} / {len(df)}")
display(longer_vqa_df)

In [ ]:
most_common_subset = print_most_common_tool_sequence_subset(longer_vqa_df)

In [ ]:

# Randomly sample 2 trajectories from the most common subset and print their JSON
import random

sampled = most_common_subset.sample(n=2, random_state=random.randint(0, 10000))

for idx, row in sampled.iterrows():
    traj_path = row["traj_path"]
    print(f"\n{'='*80}")
    print(f"Image ID: {row['image_id']}")
    print(f"Trajectory path: {traj_path}")
    print(f"{'='*80}\n")
    with open(traj_path, "r") as f:
        traj_data = json.load(f)
    assistant_messages = [m for m in traj_data if isinstance(m, dict) and m.get("role") == "assistant"]
    filtered = []
    for m in assistant_messages:
        try:
            content = json.loads(m["content"])
            entry = {}
            if "tool_name" in content:
                entry["tool_name"] = content["tool_name"]
            if "arguments" in content:
                entry["arguments"] = content["arguments"]
            if entry:
                filtered.append(entry)
        except (json.JSONDecodeError, KeyError):
            continue
    print(json.dumps(filtered, indent=2))